# Day 14 – Prompt Engineering


In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat(prompt, system=None, temperature=0.3, max_tokens=512, model="gpt-4o-mini"):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=model, max_tokens=max_tokens, temperature=temperature, messages=messages
    )
    return resp.choices[0].message.content

def show(label, prompt, **kwargs):
    preview = prompt[:120] + "..." if len(prompt) > 120 else prompt
    print(f"{'='*60}\n[{label}]\nPROMPT: {preview}")
    result = chat(prompt, **kwargs)
    print(f"RESPONSE:\n{result}\n")
    return result

print("Setup complete.")

Setup complete.


---
## Part 1 – Zero-Shot Prompting

No examples given — the model must rely purely on instructions. Works well for well-understood tasks.

In [2]:
show("1 – Zero-shot vague",
     "Classify the sentiment of this: 'The battery dies after 2 hours and support was useless.'")

[1 – Zero-shot vague]
PROMPT: Classify the sentiment of this: 'The battery dies after 2 hours and support was useless.'
RESPONSE:
The sentiment of the statement is negative.



'The sentiment of the statement is negative.'

In [3]:
show("2 – Zero-shot + output format",
     "Classify the sentiment of this review as exactly one word — Positive, Negative, or Neutral.\n"
     "Review: 'The battery dies after 2 hours and support was useless.'")

[2 – Zero-shot + output format]
PROMPT: Classify the sentiment of this review as exactly one word — Positive, Negative, or Neutral.
Review: 'The battery dies af...
RESPONSE:
Negative



'Negative'

In [4]:
show("3 – Zero-shot + persona",
     "Explain overfitting to a non-technical manager in 3 bullet points.",
     system="You are a senior data scientist known for explaining ML concepts clearly to business stakeholders.")

[3 – Zero-shot + persona]
PROMPT: Explain overfitting to a non-technical manager in 3 bullet points.
RESPONSE:
- **Too Much Detail**: Overfitting happens when a machine learning model learns the training data too well, capturing noise and random fluctuations instead of the underlying patterns. This means it becomes overly complex and specific to the training data.

- **Poor Generalization**: As a result, the model performs exceptionally well on the training data but struggles to make accurate predictions on new, unseen data. It's like a student who memorizes answers for a test but fails to understand the subject for real-world applications.

- **Balancing Act**: To avoid overfitting, we need to find a balance between complexity and simplicity in our models. This can be achieved through techniques like cross-validation, regularization, or using simpler models that focus on the most important patterns.



"- **Too Much Detail**: Overfitting happens when a machine learning model learns the training data too well, capturing noise and random fluctuations instead of the underlying patterns. This means it becomes overly complex and specific to the training data.\n\n- **Poor Generalization**: As a result, the model performs exceptionally well on the training data but struggles to make accurate predictions on new, unseen data. It's like a student who memorizes answers for a test but fails to understand the subject for real-world applications.\n\n- **Balancing Act**: To avoid overfitting, we need to find a balance between complexity and simplicity in our models. This can be achieved through techniques like cross-validation, regularization, or using simpler models that focus on the most important patterns."

In [5]:
show("4 – Zero-shot JSON output",
     "Extract the key facts from this sentence and return them as a JSON object with keys: person, action, location, time.\n"
     "Sentence: 'Elon Musk launched a rocket from Cape Canaveral on Tuesday morning.'")

[4 – Zero-shot JSON output]
PROMPT: Extract the key facts from this sentence and return them as a JSON object with keys: person, action, location, time.
Sen...
RESPONSE:
```json
{
  "person": "Elon Musk",
  "action": "launched a rocket",
  "location": "Cape Canaveral",
  "time": "Tuesday morning"
}
```



'```json\n{\n  "person": "Elon Musk",\n  "action": "launched a rocket",\n  "location": "Cape Canaveral",\n  "time": "Tuesday morning"\n}\n```'

**Zero-shot takeaways:**
- Vague prompts → verbose, inconsistent format
- Specifying output format (one word, JSON, bullets) dramatically improves reliability
- System prompts reinforce the persona throughout the conversation

---
## Part 2 – Few-Shot Prompting

Provide 2–5 input/output examples *inside the prompt* so the model learns the desired pattern without fine-tuning.

In [6]:
few_shot_sentiment = """\
Classify sentiment as Positive, Negative, or Neutral.

Review: "Absolutely love this product, works perfectly!"
Sentiment: Positive

Review: "It stopped working after a week. Very disappointed."
Sentiment: Negative

Review: "Arrived on time. Does what it says."
Sentiment: Neutral

Review: "The battery dies after 2 hours and support was useless."
Sentiment:"""

show("5 – Few-shot sentiment", few_shot_sentiment)

[5 – Few-shot sentiment]
PROMPT: Classify sentiment as Positive, Negative, or Neutral.

Review: "Absolutely love this product, works perfectly!"
Sentimen...
RESPONSE:
Negative



'Negative'

In [7]:
few_shot_ticket = """\
Convert a bug report into a structured ticket. Use this format:
Title: <short title>
Severity: Critical | High | Medium | Low
Component: <affected area>
Steps:
1. ...

---
Report: "App crashes every time I try to upload a file larger than 10MB."
Title: App crashes on large file upload
Severity: High
Component: File Upload
Steps:
1. Open the app
2. Navigate to upload section
3. Select a file > 10MB
4. Observe crash

---
Report: "Login button is invisible on dark mode but still clickable."
Title: Login button invisible in dark mode
Severity: Medium
Component: Authentication UI
Steps:
1. Enable dark mode on device
2. Open the app
3. Observe login screen — button not visible
4. Click expected button area — login proceeds normally

---
Report: "The search bar returns no results for queries with special characters like & or @."
Title:"""

show("6 – Few-shot bug ticket", few_shot_ticket)

[6 – Few-shot bug ticket]
PROMPT: Convert a bug report into a structured ticket. Use this format:
Title: <short title>
Severity: Critical | High | Medium ...
RESPONSE:
Search bar fails with special characters
Severity: High
Component: Search Functionality
Steps:
1. Open the app
2. Navigate to the search bar
3. Enter a query containing special characters (e.g., &, @)
4. Observe that no results are returned



'Search bar fails with special characters\nSeverity: High\nComponent: Search Functionality\nSteps:\n1. Open the app\n2. Navigate to the search bar\n3. Enter a query containing special characters (e.g., &, @)\n4. Observe that no results are returned'

In [8]:
few_shot_analogy = """\
Create a simple analogy that explains a technical concept to a 12-year-old.

Concept: API
Analogy: An API is like a waiter in a restaurant. You (the app) tell the waiter what you want, the waiter takes your order to the kitchen (another system), and brings back the food (data) without you ever entering the kitchen.

Concept: Encryption
Analogy: Encryption is like writing a secret note in a code only you and your friend know. Even if someone steals the note, they can't read it without the code book.

Concept: Machine learning
Analogy:"""

show("7 – Few-shot analogy", few_shot_analogy)

[7 – Few-shot analogy]
PROMPT: Create a simple analogy that explains a technical concept to a 12-year-old.

Concept: API
Analogy: An API is like a wait...
RESPONSE:
Machine learning is like teaching a dog new tricks. At first, you show the dog what to do, like sitting or rolling over. The more you practice and reward the dog when it gets it right, the better it learns to do those tricks on its own. Just like the dog learns from experience, a machine learns from data and gets better at tasks over time!



'Machine learning is like teaching a dog new tricks. At first, you show the dog what to do, like sitting or rolling over. The more you practice and reward the dog when it gets it right, the better it learns to do those tricks on its own. Just like the dog learns from experience, a machine learns from data and gets better at tasks over time!'

In [32]:
TASK = "Convert this to past tense: 'She runs to the store every morning.'"

zero = chat(TASK)

few = chat("""\
Convert to past tense.
Input: He walks to school.
Output: He walked to school.

Input: They eat lunch together.
Output: They ate lunch together.

Input: She runs to the store every morning.
Output:""")

print("Zero-shot:", zero)
print("Few-shot: ", few)

Zero-shot: 'She ran to the store every morning.'
Few-shot:  She ran to the store every morning.


**Few-shot takeaways:**
- Examples teach format AND content style simultaneously
- 2–3 diverse examples usually sufficient; more than 5 rarely helps
- Examples should cover edge cases you care about

---
## Part 3 – Chain-of-Thought (CoT) Prompting

Tell the model to **think step by step** before giving its final answer. Dramatically improves accuracy on reasoning tasks.

In [10]:
MATH_PROBLEM = (
    "A store sells apples for Rs 12 each and bananas for Rs 8 each. "
    "Rahul buys 5 apples and 3 bananas, and pays with a Rs 100 note. "
    "How much change does he receive?"
)

direct = chat(MATH_PROBLEM)
print("[9 – Direct answer]")
print(direct)

[9 – Direct answer]
To find out how much change Rahul receives, we first need to calculate the total cost of the apples and bananas he buys.

1. **Cost of apples**: Rahul buys 5 apples at Rs 12 each.
   \[
   \text{Cost of apples} = 5 \times 12 = Rs 60
   \]

2. **Cost of bananas**: Rahul buys 3 bananas at Rs 8 each.
   \[
   \text{Cost of bananas} = 3 \times 8 = Rs 24
   \]

3. **Total cost**: Now, we add the cost of apples and bananas together.
   \[
   \text{Total cost} = \text{Cost of apples} + \text{Cost of bananas} = 60 + 24 = Rs 84
   \]

4. **Change received**: Rahul pays with a Rs 100 note, so we subtract the total cost from the amount he paid.
   \[
   \text{Change} = 100 - 84 = Rs 16
   \]

Thus, Rahul receives Rs 16 as change.


In [11]:
cot = chat(MATH_PROBLEM + "\n\nThink step by step.")
print("[10 – Zero-shot CoT]")
print(cot)

[10 – Zero-shot CoT]
To find out how much change Rahul receives, we first need to calculate the total cost of the apples and bananas he buys.

1. **Calculate the cost of apples:**
   - Rahul buys 5 apples.
   - The price of each apple is Rs 12.
   - Total cost for apples = Number of apples × Price per apple
   \[
   \text{Total cost for apples} = 5 \times 12 = 60 \text{ Rs}
   \]

2. **Calculate the cost of bananas:**
   - Rahul buys 3 bananas.
   - The price of each banana is Rs 8.
   - Total cost for bananas = Number of bananas × Price per banana
   \[
   \text{Total cost for bananas} = 3 \times 8 = 24 \text{ Rs}
   \]

3. **Calculate the total cost of the purchase:**
   - Total cost = Total cost for apples + Total cost for bananas
   \[
   \text{Total cost} = 60 + 24 = 84 \text{ Rs}
   \]

4. **Calculate the change received:**
   - Rahul pays with a Rs 100 note.
   - Change = Amount paid - Total cost
   \[
   \text{Change} = 100 - 84 = 16 \text{ Rs}
   \]

Therefore, Rahul receives 

In [12]:
few_shot_cot = """\
Solve word problems step by step, then state the final answer.

Problem: A train travels at 60 km/h for 2 hours and then at 80 km/h for 1 hour. What is the total distance?
Step 1: Distance in first part = 60 × 2 = 120 km
Step 2: Distance in second part = 80 × 1 = 80 km
Step 3: Total distance = 120 + 80 = 200 km
Answer: 200 km

Problem: A store sells apples for Rs 12 each and bananas for Rs 8 each. Rahul buys 5 apples and 3 bananas, and pays with a Rs 100 note. How much change does he receive?
"""
show("11 – Few-shot CoT", few_shot_cot)

[11 – Few-shot CoT]
PROMPT: Solve word problems step by step, then state the final answer.

Problem: A train travels at 60 km/h for 2 hours and then...
RESPONSE:
Let's solve the problem step by step.

**Step 1:** Calculate the cost of the apples.
- Rahul buys 5 apples at Rs 12 each.
- Cost of apples = 5 × 12 = Rs 60.

**Step 2:** Calculate the cost of the bananas.
- Rahul buys 3 bananas at Rs 8 each.
- Cost of bananas = 3 × 8 = Rs 24.

**Step 3:** Calculate the total cost of the fruits.
- Total cost = Cost of apples + Cost of bananas = 60 + 24 = Rs 84.

**Step 4:** Calculate the change Rahul receives after paying with a Rs 100 note.
- Change = Amount paid - Total cost = 100 - 84 = Rs 16.

**Final Answer:** Rahul receives Rs 16 as change.



"Let's solve the problem step by step.\n\n**Step 1:** Calculate the cost of the apples.\n- Rahul buys 5 apples at Rs 12 each.\n- Cost of apples = 5 × 12 = Rs 60.\n\n**Step 2:** Calculate the cost of the bananas.\n- Rahul buys 3 bananas at Rs 8 each.\n- Cost of bananas = 3 × 8 = Rs 24.\n\n**Step 3:** Calculate the total cost of the fruits.\n- Total cost = Cost of apples + Cost of bananas = 60 + 24 = Rs 84.\n\n**Step 4:** Calculate the change Rahul receives after paying with a Rs 100 note.\n- Change = Amount paid - Total cost = 100 - 84 = Rs 16.\n\n**Final Answer:** Rahul receives Rs 16 as change."

In [13]:
buggy_code = """\
This Python function is supposed to return the average of a list, but it's wrong.
Find the bug and fix it. Think step by step.

```python
def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums) - 1
```
"""
show("12 – CoT debugging", buggy_code)

[12 – CoT debugging]
PROMPT: This Python function is supposed to return the average of a list, but it's wrong.
Find the bug and fix it. Think step by...
RESPONSE:
Let's analyze the provided function step by step to identify the bug and fix it.

### Step 1: Understanding the Function
The function `average(nums)` is intended to calculate the average of a list of numbers. The average is defined as the sum of the numbers divided by the count of the numbers.

### Step 2: Identifying the Bug
The line that calculates the average is:
```python
return total / len(nums) - 1
```
Here, the issue is with the `- 1` at the end. This incorrectly subtracts 1 from the average, which is not how averages are calculated. The correct formula should simply be `total / len(nums)`.

### Step 3: Fixing the Bug
We need to remove the `- 1` from the return statement. Additionally, we should also handle the case where the input list `nums` might be empty to avoid a division by zero error.

### Step 4: Implementing 

"Let's analyze the provided function step by step to identify the bug and fix it.\n\n### Step 1: Understanding the Function\nThe function `average(nums)` is intended to calculate the average of a list of numbers. The average is defined as the sum of the numbers divided by the count of the numbers.\n\n### Step 2: Identifying the Bug\nThe line that calculates the average is:\n```python\nreturn total / len(nums) - 1\n```\nHere, the issue is with the `- 1` at the end. This incorrectly subtracts 1 from the average, which is not how averages are calculated. The correct formula should simply be `total / len(nums)`.\n\n### Step 3: Fixing the Bug\nWe need to remove the `- 1` from the return statement. Additionally, we should also handle the case where the input list `nums` might be empty to avoid a division by zero error.\n\n### Step 4: Implementing the Fix\nHere’s the corrected version of the function:\n\n```python\ndef average(nums):\n    if len(nums) == 0:  # Check for empty list\n        re

In [14]:
decision = """\
I have Rs 50,000 to invest. Options are:
A) Fixed deposit at 7% annual interest
B) Mutual fund with expected 12% return but 15% risk of losing 20%
C) Gold which historically gives 8% returns with low volatility

I am 25 years old with a stable job and no dependents.
Which should I choose? Think through each option step by step before recommending.
"""
show("13 – CoT decision", decision)

[13 – CoT decision]
PROMPT: I have Rs 50,000 to invest. Options are:
A) Fixed deposit at 7% annual interest
B) Mutual fund with expected 12% return ...
RESPONSE:
When considering your investment options, it's essential to evaluate each one based on potential returns, risks, and your personal financial situation. Let's break down each option step by step:

### Option A: Fixed Deposit at 7% Annual Interest
- **Returns**: A fixed deposit (FD) offers a guaranteed return of 7% per annum. If you invest Rs 50,000 for one year, you would earn Rs 3,500 in interest, bringing your total to Rs 53,500.
- **Risk**: The risk is minimal since the principal amount is secure, and the interest rate is fixed. FDs are typically insured up to a certain limit by the government.
- **Liquidity**: FDs may have a lock-in period, which means your money could be tied up for a specific duration, making it less liquid.
- **Inflation**: The real return after accounting for inflation may be lower than 7%, depending on

"When considering your investment options, it's essential to evaluate each one based on potential returns, risks, and your personal financial situation. Let's break down each option step by step:\n\n### Option A: Fixed Deposit at 7% Annual Interest\n- **Returns**: A fixed deposit (FD) offers a guaranteed return of 7% per annum. If you invest Rs 50,000 for one year, you would earn Rs 3,500 in interest, bringing your total to Rs 53,500.\n- **Risk**: The risk is minimal since the principal amount is secure, and the interest rate is fixed. FDs are typically insured up to a certain limit by the government.\n- **Liquidity**: FDs may have a lock-in period, which means your money could be tied up for a specific duration, making it less liquid.\n- **Inflation**: The real return after accounting for inflation may be lower than 7%, depending on current inflation rates.\n\n### Option B: Mutual Fund with Expected 12% Return but 15% Risk of Losing 20%\n- **Returns**: The mutual fund has an expected 

**CoT takeaways:**
- "Think step by step" reliably improves accuracy on math and logic tasks
- Few-shot CoT outperforms zero-shot CoT on complex multi-step problems
- CoT increases output length — set `max_tokens` accordingly
- For pure extraction/classification tasks, CoT overhead is usually not worth it

---
## Part 4 – Summarization Prompts

In [15]:
LONG_TEXT = """\
India's space agency ISRO successfully launched the Chandrayaan-3 mission on July 14, 2023.
The spacecraft consisted of a lander named Vikram and a rover named Pragyan.
After a journey of about 40 days, Chandrayaan-3 made a historic soft landing near the lunar
south pole on August 23, 2023. India became the fourth country to land on the Moon and the
first to land near the south pole. The south pole region is scientifically significant because
it is believed to contain water ice in permanently shadowed craters. The Pragyan rover deployed
from Vikram and collected data on soil composition and temperature. ISRO scientists confirmed
the presence of sulphur on the lunar surface for the first time. The mission cost approximately
Rs 615 crore, far less than many other space missions globally, demonstrating India's ability
to conduct cost-effective deep space exploration.
"""

In [16]:
show("14 – Generic summary", f"Summarize this:\n\n{LONG_TEXT}")

[14 – Generic summary]
PROMPT: Summarize this:

India's space agency ISRO successfully launched the Chandrayaan-3 mission on July 14, 2023.
The spacecr...
RESPONSE:
India's space agency ISRO successfully launched the Chandrayaan-3 mission on July 14, 2023, which included a lander named Vikram and a rover named Pragyan. After a 40-day journey, the mission achieved a historic soft landing near the lunar south pole on August 23, 2023, making India the fourth country to land on the Moon and the first to do so near the south pole, a region believed to contain water ice. The Pragyan rover collected data on soil composition and temperature, confirming the presence of sulphur on the lunar surface for the first time. The mission cost approximately Rs 615 crore, showcasing India's capability for cost-effective deep space exploration.



"India's space agency ISRO successfully launched the Chandrayaan-3 mission on July 14, 2023, which included a lander named Vikram and a rover named Pragyan. After a 40-day journey, the mission achieved a historic soft landing near the lunar south pole on August 23, 2023, making India the fourth country to land on the Moon and the first to do so near the south pole, a region believed to contain water ice. The Pragyan rover collected data on soil composition and temperature, confirming the presence of sulphur on the lunar surface for the first time. The mission cost approximately Rs 615 crore, showcasing India's capability for cost-effective deep space exploration."

In [17]:
show("15 – Constrained summary",
     f"Summarize the following for a 10-year-old in exactly 2 sentences:\n\n{LONG_TEXT}")

[15 – Constrained summary]
PROMPT: Summarize the following for a 10-year-old in exactly 2 sentences:

India's space agency ISRO successfully launched the C...
RESPONSE:
India's space agency, ISRO, launched a mission called Chandrayaan-3, which included a lander named Vikram and a rover named Pragyan, and they successfully landed on the Moon's south pole on August 23, 2023. This mission was special because it was the first time anyone landed there, and the rover found important information, like the presence of sulphur and the possibility of water ice!



"India's space agency, ISRO, launched a mission called Chandrayaan-3, which included a lander named Vikram and a rover named Pragyan, and they successfully landed on the Moon's south pole on August 23, 2023. This mission was special because it was the first time anyone landed there, and the rover found important information, like the presence of sulphur and the possibility of water ice!"

In [18]:
show("16 – Bullet key facts",
     f"Extract the 5 most important facts from this passage as a bullet list. Be concise.\n\n{LONG_TEXT}")

[16 – Bullet key facts]
PROMPT: Extract the 5 most important facts from this passage as a bullet list. Be concise.

India's space agency ISRO successful...
RESPONSE:
- ISRO launched the Chandrayaan-3 mission on July 14, 2023.
- The spacecraft included a lander named Vikram and a rover named Pragyan.
- Chandrayaan-3 achieved a soft landing near the lunar south pole on August 23, 2023.
- India became the fourth country to land on the Moon and the first near the south pole.
- The mission confirmed the presence of sulphur on the lunar surface and cost approximately Rs 615 crore.



'- ISRO launched the Chandrayaan-3 mission on July 14, 2023.\n- The spacecraft included a lander named Vikram and a rover named Pragyan.\n- Chandrayaan-3 achieved a soft landing near the lunar south pole on August 23, 2023.\n- India became the fourth country to land on the Moon and the first near the south pole.\n- The mission confirmed the presence of sulphur on the lunar surface and cost approximately Rs 615 crore.'

In [19]:
show("17 – TLDR + headline",
     f"Read the passage and respond with:\nHeadline: <newspaper-style headline, max 12 words>\nTL;DR: <one sentence, max 25 words>\n\n{LONG_TEXT}")

[17 – TLDR + headline]
PROMPT: Read the passage and respond with:
Headline: <newspaper-style headline, max 12 words>
TL;DR: <one sentence, max 25 words...
RESPONSE:
Headline: India Achieves Historic Moon Landing with Chandrayaan-3 Mission

TL;DR: ISRO's Chandrayaan-3 successfully landed near the Moon's south pole, marking India's historic achievement in space exploration.



"Headline: India Achieves Historic Moon Landing with Chandrayaan-3 Mission\n\nTL;DR: ISRO's Chandrayaan-3 successfully landed near the Moon's south pole, marking India's historic achievement in space exploration."

---
## Part 5 – Extraction Prompts

In [20]:
INVOICE_TEXT = """\
INVOICE #2024-0892
Date: March 15, 2024
Bill To: Acme Corp, 123 Main St, Mumbai 400001
Contact: Ramesh Sharma (ramesh@acme.com)

Items:
- Web Development Services (40 hrs @ Rs 2,500/hr): Rs 1,00,000
- UI/UX Design (15 hrs @ Rs 3,000/hr): Rs 45,000
- Domain & Hosting (annual): Rs 8,500

Subtotal: Rs 1,53,500
GST (18%): Rs 27,630
Total Due: Rs 1,81,130
Due Date: April 14, 2024
Payment: Bank transfer to HDFC Account 1234567890 IFSC HDFC0001234
"""

In [21]:
show("18 – Named entity extraction",
     f"Extract all named entities from this invoice. Group them by type: Person, Organization, Location, Date, Amount.\n\n{INVOICE_TEXT}")

[18 – Named entity extraction]
PROMPT: Extract all named entities from this invoice. Group them by type: Person, Organization, Location, Date, Amount.

INVOICE...
RESPONSE:
Here are the extracted named entities from the invoice, grouped by type:

**Person:**
- Ramesh Sharma

**Organization:**
- Acme Corp
- HDFC

**Location:**
- Mumbai

**Date:**
- March 15, 2024
- April 14, 2024

**Amount:**
- Rs 1,00,000
- Rs 45,000
- Rs 8,500
- Rs 1,53,500
- Rs 27,630
- Rs 1,81,130



'Here are the extracted named entities from the invoice, grouped by type:\n\n**Person:**\n- Ramesh Sharma\n\n**Organization:**\n- Acme Corp\n- HDFC\n\n**Location:**\n- Mumbai\n\n**Date:**\n- March 15, 2024\n- April 14, 2024\n\n**Amount:**\n- Rs 1,00,000\n- Rs 45,000\n- Rs 8,500\n- Rs 1,53,500\n- Rs 27,630\n- Rs 1,81,130'

In [22]:
show("19 – JSON extraction",
     f"Extract invoice data as JSON with these keys: invoice_number, date, client_name, client_email, items (array of name/amount), total_due, due_date. Return only valid JSON.\n\n{INVOICE_TEXT}")

[19 – JSON extraction]
PROMPT: Extract invoice data as JSON with these keys: invoice_number, date, client_name, client_email, items (array of name/amou...
RESPONSE:
```json
{
  "invoice_number": "2024-0892",
  "date": "March 15, 2024",
  "client_name": "Acme Corp",
  "client_email": "ramesh@acme.com",
  "items": [
    {
      "name": "Web Development Services",
      "amount": 100000
    },
    {
      "name": "UI/UX Design",
      "amount": 45000
    },
    {
      "name": "Domain & Hosting",
      "amount": 8500
    }
  ],
  "total_due": 181130,
  "due_date": "April 14, 2024"
}
```



'```json\n{\n  "invoice_number": "2024-0892",\n  "date": "March 15, 2024",\n  "client_name": "Acme Corp",\n  "client_email": "ramesh@acme.com",\n  "items": [\n    {\n      "name": "Web Development Services",\n      "amount": 100000\n    },\n    {\n      "name": "UI/UX Design",\n      "amount": 45000\n    },\n    {\n      "name": "Domain & Hosting",\n      "amount": 8500\n    }\n  ],\n  "total_due": 181130,\n  "due_date": "April 14, 2024"\n}\n```'

In [23]:
show("20 – Conditional extraction",
     f"From this invoice, extract only line items that cost more than Rs 50,000. Format as a simple list.\n\n{INVOICE_TEXT}")

[20 – Conditional extraction]
PROMPT: From this invoice, extract only line items that cost more than Rs 50,000. Format as a simple list.

INVOICE #2024-0892
D...
RESPONSE:
- Web Development Services: Rs 1,00,000



'- Web Development Services: Rs 1,00,000'

---
## Part 6 – Generation Prompts

In [24]:
show("21 – Email vague",
     "Write an email asking for a meeting.")

[21 – Email vague]
PROMPT: Write an email asking for a meeting.
RESPONSE:
Subject: Request for Meeting

Dear [Recipient's Name],

I hope this message finds you well. I am writing to request a meeting at your earliest convenience to discuss [briefly state the purpose or topic of the meeting, e.g., "our upcoming project," "collaboration opportunities," or "recent developments in our work"].

I believe that a meeting would be beneficial for us to align our goals and explore potential strategies moving forward. Please let me know your availability in the coming days, and I will do my best to accommodate.

Thank you for considering my request. I look forward to your response.

Best regards,

[Your Name]  
[Your Position]  
[Your Company]  
[Your Phone Number]  
[Your Email Address]  



'Subject: Request for Meeting\n\nDear [Recipient\'s Name],\n\nI hope this message finds you well. I am writing to request a meeting at your earliest convenience to discuss [briefly state the purpose or topic of the meeting, e.g., "our upcoming project," "collaboration opportunities," or "recent developments in our work"].\n\nI believe that a meeting would be beneficial for us to align our goals and explore potential strategies moving forward. Please let me know your availability in the coming days, and I will do my best to accommodate.\n\nThank you for considering my request. I look forward to your response.\n\nBest regards,\n\n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Your Phone Number]  \n[Your Email Address]  '

In [33]:
show("22 – Email specific",
     """\
Write a professional email from Pradhnyesh (ML intern at Linkific) to his manager Aditya
requesting a 10-minute meeting tomorrow to discuss the Day-14 tasks.
Tone: friendly but professional. Length: under 100 words.
""")

[22 – Email specific]
PROMPT: Write a professional email from Pradhnyesh (ML intern at Linkific) to his manager Aditya
requesting a 10-minute meeting ...
RESPONSE:
Subject: Request for a Quick Meeting Tomorrow

Hi Aditya,

I hope this message finds you well. I would like to request a brief 10-minute meeting tomorrow to discuss the Day-14 tasks and ensure I’m on the right track. Please let me know a time that works for you.

Thank you for your support!

Best regards,  
Pradhnyesh  
ML Intern, Linkific



'Subject: Request for a Quick Meeting Tomorrow\n\nHi Aditya,\n\nI hope this message finds you well. I would like to request a brief 10-minute meeting tomorrow to discuss the Day-14 tasks and ensure I’m on the right track. Please let me know a time that works for you.\n\nThank you for your support!\n\nBest regards,  \nPradhnyesh  \nML Intern, Linkific'

In [26]:
show("23 – Code generation",
     """\
Write a Python function `retry(func, max_attempts=3, delay_seconds=1)` that:
- Calls func()
- If it raises an exception, waits delay_seconds and tries again
- After max_attempts failures, re-raises the last exception
- Logs each attempt with print()
Include a usage example at the bottom. No docstring needed.
""")

[23 – Code generation]
PROMPT: Write a Python function `retry(func, max_attempts=3, delay_seconds=1)` that:
- Calls func()
- If it raises an exception,...
RESPONSE:
```python
import time

def retry(func, max_attempts=3, delay_seconds=1):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                time.sleep(delay_seconds)
    raise e

# Usage example
def unreliable_function():
    import random
    if random.random() < 0.7:  # 70% chance to raise an exception
        raise ValueError("Random failure!")
    return "Success!"

if __name__ == "__main__":
    try:
        result = retry(unreliable_function, max_attempts=5, delay_seconds=2)
        print(result)
    except Exception as final_exception:
        print(f"Function failed after all attempts: {final_exception}")
```



'```python\nimport time\n\ndef retry(func, max_attempts=3, delay_seconds=1):\n    for attempt in range(1, max_attempts + 1):\n        try:\n            return func()\n        except Exception as e:\n            print(f"Attempt {attempt} failed: {e}")\n            if attempt < max_attempts:\n                time.sleep(delay_seconds)\n    raise e\n\n# Usage example\ndef unreliable_function():\n    import random\n    if random.random() < 0.7:  # 70% chance to raise an exception\n        raise ValueError("Random failure!")\n    return "Success!"\n\nif __name__ == "__main__":\n    try:\n        result = retry(unreliable_function, max_attempts=5, delay_seconds=2)\n        print(result)\n    except Exception as final_exception:\n        print(f"Function failed after all attempts: {final_exception}")\n```'

In [27]:
show("24 – Creative generation",
     """\
Write a product description for a mechanical keyboard aimed at programmers.
Style: like an Apple product page — short punchy sentences, present tense, no exclamation marks.
Length: 60–80 words.
""",
     temperature=0.9)

[24 – Creative generation]
PROMPT: Write a product description for a mechanical keyboard aimed at programmers.
Style: like an Apple product page — short pu...
RESPONSE:
Experience precision with the CodeKey Pro Mechanical Keyboard. Engineered for programmers, it features tactile switches that deliver satisfying feedback with every keystroke. The sleek, minimalist design fits seamlessly into any workspace. Customizable RGB lighting enhances your coding environment while ergonomic contours support long hours at the keyboard. With programmable keys and on-the-fly macro support, optimize your workflow effortlessly. Elevate your coding experience—unlock your potential with CodeKey Pro.



'Experience precision with the CodeKey Pro Mechanical Keyboard. Engineered for programmers, it features tactile switches that deliver satisfying feedback with every keystroke. The sleek, minimalist design fits seamlessly into any workspace. Customizable RGB lighting enhances your coding environment while ergonomic contours support long hours at the keyboard. With programmable keys and on-the-fly macro support, optimize your workflow effortlessly. Elevate your coding experience—unlock your potential with CodeKey Pro.'

In [28]:
show("25 – Test data generation",
     """\
Generate 5 realistic but fake customer records as a JSON array. Each record must have:
customer_id (UUID v4 format), name (Indian name), email, phone (Indian format),
city (Indian city), plan (Free | Basic | Pro), created_at (ISO 8601, between 2023-01-01 and 2024-12-31).
Return only the JSON array, no explanation.
""")

[25 – Test data generation]
PROMPT: Generate 5 realistic but fake customer records as a JSON array. Each record must have:
customer_id (UUID v4 format), nam...
RESPONSE:
```json
[
    {
        "customer_id": "c9b1e2d4-6b6e-4d3b-9c7e-9f7f5e3c5c2a",
        "name": "Aarav Sharma",
        "email": "aarav.sharma@example.com",
        "phone": "+91 98765 43210",
        "city": "Bangalore",
        "plan": "Pro",
        "created_at": "2023-05-15T10:30:00Z"
    },
    {
        "customer_id": "f4e2a1b3-5c1f-4c5b-8c4b-8a4f3d2e1b6e",
        "name": "Diya Patel",
        "email": "diya.patel@example.com",
        "phone": "+91 87654 32109",
        "city": "Mumbai",
        "plan": "Basic",
        "created_at": "2023-08-22T14:45:00Z"
    },
    {
        "customer_id": "a1b2c3d4-e5f6-7g8h-9i0j-k1l2m3n4o5p6",
        "name": "Rohan Verma",
        "email": "rohan.verma@example.com",
        "phone": "+91 76543 21098",
        "city": "Delhi",
        "plan": "Free",
        "created_at": "2

'```json\n[\n    {\n        "customer_id": "c9b1e2d4-6b6e-4d3b-9c7e-9f7f5e3c5c2a",\n        "name": "Aarav Sharma",\n        "email": "aarav.sharma@example.com",\n        "phone": "+91 98765 43210",\n        "city": "Bangalore",\n        "plan": "Pro",\n        "created_at": "2023-05-15T10:30:00Z"\n    },\n    {\n        "customer_id": "f4e2a1b3-5c1f-4c5b-8c4b-8a4f3d2e1b6e",\n        "name": "Diya Patel",\n        "email": "diya.patel@example.com",\n        "phone": "+91 87654 32109",\n        "city": "Mumbai",\n        "plan": "Basic",\n        "created_at": "2023-08-22T14:45:00Z"\n    },\n    {\n        "customer_id": "a1b2c3d4-e5f6-7g8h-9i0j-k1l2m3n4o5p6",\n        "name": "Rohan Verma",\n        "email": "rohan.verma@example.com",\n        "phone": "+91 76543 21098",\n        "city": "Delhi",\n        "plan": "Free",\n        "created_at": "2024-01-10T09:15:00Z"\n    },\n    {\n        "customer_id": "d7e8f9a0-b1c2-3d4e-5f6g-7h8i9j0k1l2m",\n        "name": "Sneha Iyer",\n        "e

---
## Part 7 – Technique Comparison Matrix

In [29]:
import pandas as pd

comparison = {
    "Technique":      ["Zero-shot",    "Zero-shot + format", "Few-shot",        "Chain-of-Thought", "CoT + Few-shot"],
    "Setup effort":   ["None",         "Low",                "Medium",          "Low",              "Medium"],
    "Best for":       ["Simple tasks", "Structured output",  "Pattern tasks",   "Reasoning",        "Complex reasoning"],
    "Token cost":     ["Low",          "Low",                "Medium",          "High",             "High"],
    "Accuracy":       ["Low-Medium",   "Medium",             "Medium-High",     "High",             "Highest"],
    "Consistency":    ["Low",          "Medium",             "High",            "Medium",           "High"],
}

df = pd.DataFrame(comparison)
df

,Technique,Setup effort,Best for,Token cost,Accuracy,Consistency
0,Zero-shot,None,Simple tasks,Low,Low-Medium,Low
1,Zero-shot + format,Low,Structured output,Low,Medium,Medium
2,Few-shot,Medium,Pattern tasks,Medium,Medium-High,High
3,Chain-of-Thought,Low,Reasoning,High,High,Medium
4,CoT + Few-shot,Medium,Complex reasoning,High,Highest,High


## Part 8 – The 10 Prompts Rule

For any task, write multiple variations and measure which works best.  
Here we test **sentiment classification** with 5 variants on the same sentence.

In [30]:
TARGET = "The delivery was two days late but the packaging was excellent and the product works great."

prompts = {
    "P1 – Bare": f'Sentiment of: "{TARGET}"',

    "P2 – Label constrained": (
        f'Classify as Positive, Negative, or Mixed. Reply with one word only.\n"{TARGET}"'
    ),
    "P3 – Score": (
        f'Rate the sentiment of this review on a scale of 1 (very negative) to 5 (very positive). '
        f'Reply with a number only.\n"{TARGET}"'
    ),
    "P4 – Aspect-based": (
        f'For each aspect in the review, label sentiment as Positive/Negative/Neutral. '
        f'Format: aspect: sentiment\n"{TARGET}"'
    ),
    "P5 – CoT": (
        f'Analyze this review step by step: identify positives, identify negatives, '
        f'then give an overall sentiment (Positive/Negative/Mixed).\n"{TARGET}"'
    ),
}

for label, p in prompts.items():
    result = chat(p, temperature=0)
    print(f"\n[{label}]")
    print(result)


[P1 – Bare]
The sentiment of the statement is mixed. While it expresses a negative sentiment regarding the delivery being late, it also conveys positive sentiments about the packaging and the product's performance. Overall, the positive aspects somewhat balance out the negative, but the late delivery is a notable concern.

[P2 – Label constrained]
Mixed

[P3 – Score]
4

[P4 – Aspect-based]
delivery: Negative  
packaging: Positive  
product: Positive

[P5 – CoT]
Let's analyze the review step by step:

### Positives:
1. **Excellent Packaging**: The reviewer appreciates the quality of the packaging, indicating that it was well done and likely protected the product during transit.
2. **Product Works Great**: The reviewer is satisfied with the product's performance, suggesting that it meets or exceeds their expectations.

### Negatives:
1. **Delivery Delay**: The reviewer mentions that the delivery was two days late, which is a significant inconvenience and could affect their overall exper

In [31]:
print("""
Observations:
- P1 (bare): Verbose, inconsistent format — hard to parse programmatically
- P2 (label constrained): Single word, consistent — best for classification pipelines
- P3 (score): Numeric output, easy to average over many reviews
- P4 (aspect-based): Most informative — captures nuance (late delivery vs good product)
- P5 (CoT): Best for nuanced edge cases, but 5x more tokens than P2

Winner depends on use case:
  → Dashboard filter: P2
  → Rating aggregation: P3
  → Product feedback analysis: P4
  → Customer complaint routing: P5
""")


Observations:
- P1 (bare): Verbose, inconsistent format — hard to parse programmatically
- P2 (label constrained): Single word, consistent — best for classification pipelines
- P3 (score): Numeric output, easy to average over many reviews
- P4 (aspect-based): Most informative — captures nuance (late delivery vs good product)
- P5 (CoT): Best for nuanced edge cases, but 5x more tokens than P2

Winner depends on use case:
  → Dashboard filter: P2
  → Rating aggregation: P3
  → Product feedback analysis: P4
  → Customer complaint routing: P5

